# 1. Import and Hardware Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt
import numpy as np
import copy

!pip install tqdm -q
from tqdm.auto import tqdm

!pip install wandb -q
import wandb

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
DATA_PATH = './Data'
SAVE_PATH = '.Model'

In [ ]:
wandb.login()

# 2. Hyperparameters

In [ ]:
IMG_SIZE = 224
IN_CHANNELS = 3
BATCH_SIZE = 64 # GoogLeNet is slightly deeper/complex, adjusting batch size if needed, but 64/128 usually fits

EPOCHS = 10
LR = 0.01
NUM_CLS = 101
DROPOUT_RATE = 0.4

# 3. Training Data Preparation

In [ ]:
# Mean and Std
stats = ((0.545, 0.443, 0.344), (0.269, 0.271, 0.276))

# Transform for train data
train_transform = transforms.Compose([
    transforms.Resize(265),
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.ColorJitter(),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
    transforms.RandomErasing(p=0.2)
])

# Transform for test data
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [ ]:
# Download the full train data
full_train_data = datasets.Food101(root=DATA_PATH, split='train', 
                        download=True, transform=train_transform)

# Split the full train data into train and validation subset
train_size = int(0.8 * len(full_train_data))
val_size = len(full_train_data) - train_size
train_subset, val_subset = random_split(full_train_data, [train_size, val_size])

# Change the transform of val subset to test_transform
val_subset.dataset = copy.copy(full_train_data)
val_subset.dataset.transform = test_transform

# Download the test data
test_data = datasets.Food101(root=DATA_PATH, split='test', 
                        download=True, transform=test_transform)

In [ ]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

# 4. Model Architecture (GoogLeNet)

In [ ]:
class Inception(nn.Module):
    def __init__(self, in_channels, ch1x1, ch3x3red, ch3x3, ch5x5red, ch5x5, pool_proj):
        super(Inception, self).__init__()

        # 1x1 conv branch
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1),
            nn.ReLU(inplace=True)
        )

        # 1x1 conv -> 3x3 conv branch
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3red, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch3x3red, ch3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        # 1x1 conv -> 5x5 conv branch
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5red, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch5x5red, ch5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )

        # 3x3 pool -> 1x1 conv branch
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        branch1_output = self.branch1(x)
        branch2_output = self.branch2(x)
        branch3_output = self.branch3(x)
        branch4_output = self.branch4(x)

        return torch.cat([branch1_output, branch2_output, branch3_output, branch4_output], 1)


class GoogLeNet(nn.Module):
    def __init__(self, num_cls=1000, dropout_rate=0.4):
        super(GoogLeNet, self).__init__()

        self.pre_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.LocalResponseNorm(2),
            nn.Conv2d(64, 64, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(2),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Inception blocks
        self.a3 = Inception(192, 64, 96, 128, 16, 32, 32)
        self.b3 = Inception(256, 128, 128, 192, 32, 96, 64)

        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.a4 = Inception(480, 192, 96, 208, 16, 48, 64)
        self.b4 = Inception(512, 160, 112, 224, 24, 64, 64)
        self.c4 = Inception(512, 128, 128, 256, 24, 64, 64)
        self.d4 = Inception(512, 112, 144, 288, 32, 64, 64)
        self.e4 = Inception(528, 256, 160, 320, 32, 128, 128)

        self.a5 = Inception(832, 256, 160, 320, 32, 128, 128)
        self.b5 = Inception(832, 384, 192, 384, 48, 128, 128)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc = nn.Linear(1024, num_cls)

    def forward(self, x):
        x = self.pre_layers(x)
        x = self.a3(x)
        x = self.b3(x)
        x = self.maxpool(x)
        x = self.a4(x)
        x = self.b4(x)
        x = self.c4(x)
        x = self.d4(x)
        x = self.e4(x)
        x = self.maxpool(x)
        x = self.a5(x)
        x = self.b5(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = GoogLeNet(num_cls=NUM_CLS, dropout_rate=DROPOUT_RATE).to(device)

# 5. Training Preparation

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta = 0, verbose = False, save_path='best_model.pth', trace_func=print):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.save_path = save_path
        self.trace_func = trace_func

        self.best_score = None
        self.earlystop = False
        self.counter = 0

    def __call__(self, val_loss, model):

        # The first epoch
        if self.best_score is None:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)

        # The loss didnt reduce much as expect or increased
        elif val_loss >= self.best_score - self.delta:
            self.counter += 1
            self.trace_func(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if (self.counter >= self.patience):
                self.earlystop = True
        
        # The loss reduced properly
        else:
            self.best_score = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        
        if self.verbose:
            self.trace_func("Best Model saving ...")
        torch.save(model.state_dict(), self.save_path)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience= 5, factor=0.1)

# Using GradScaler to prevent Gradient Underflow by using FP16
scaler = torch.amp.GradScaler(device)

In [ ]:
def train(model, loader, criterion, optimizer, scaler):
    # Set the Model in Training Mode
    model.train()

    loop = tqdm(loader, desc="Training", leave=False)
    train_loss, train_acc = 0, 0

    for x, y in loop:
        # Move data to device
        x, y = x.to(device), y.to(device)

        # Clear the Gradient of last batch
        optimizer.zero_grad(set_to_none=True)

        # Get prediction & loss with Automatic Mixed Precision (AMP)
        with torch.amp.autocast(device_type=device.type):
            out = model(x)
            loss = criterion(out, y)

        # Scale up the loss and backpropagate
        scaler.scale(loss).backward()

        # Scale down the Gradients before clipping
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Check if the gradients are valid after Unscaling
        scaler.step(optimizer)

        # Adjust the Scale Factor for the next batch
        scaler.update()

        # Sum up the loss and accuracy
        train_loss += loss.item() * x.size(0)
        train_acc += (out.argmax(1) == y).sum().item()

    return train_loss / len(loader.dataset), train_acc / len(loader.dataset)

In [ ]:
import torch.profiler as profiler

def train_with_profiling(model, loader, criterion, optimizer, scaler):
    # Set the Model in Training Mode
    model.train()
    processed_samples = 0

    # Set up for the profiler
    activities=[profiler.ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(profiler.ProfilerActivity.CUDA)

    with profiler.profile(
        activities=activities,
        record_shapes = True,
        profile_memory=True,
        with_stack=True
    ) as prof:
        loop = tqdm(loader, desc="Profiling", leave=False)

        for i, (x, y) in enumerate(loop):
            # Move data to device
            x, y = x.to(device), y.to(device)

            # Clear the Gradient of last batch
            optimizer.zero_grad(set_to_none=True)

            # Get prediction & loss with Automatic Mixed Precision (AMP)
            with torch.amp.autocast(device_type=device.type):
                out = model(x)
                loss = criterion(out, y)

            # Scale up the loss and backpropagate
            scaler.scale(loss).backward()

            # Scale down the Gradients before clipping
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Check if the gradients are valid after Unscaling
            scaler.step(optimizer)

            # Adjust the Scale Factor for the next batch
            scaler.update()

            processed_samples += x.size(0)

            prof.step()

            if i >= 5:
                break

    print("\n" + "="*50)
    print("PROFILER: TOP OPERATIONS")
    print("="*50)
    # print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

    print("\n" + "="*50)
    print("PROFILER: MEMORY USAGE")
    print("="*50)
    print(prof.key_averages().table(sort_by="self_cpu_memory_usage", row_limit=10))

    return "Profiling beendet."

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    loop = tqdm(loader, desc="Validation", leave=False)
    val_loss, val_acc = 0, 0

    for x, y in loop:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            out = model(x)
            loss = criterion(out, y)

        val_loss += loss.item() * x.size(0)
        val_acc += (out.argmax(1) == y).sum().item()
    
    return val_loss / len(loader.dataset), val_acc / len(loader.dataset)

In [ ]:
def test(model, loader):
    model.eval()
    test_acc = 0
    loop = tqdm(loader, desc="Testing", leave=False)
    for x, y in loop:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            out = model(x)
        test_acc += (out.argmax(1) == y).sum().item()

    return test_acc / len(loader.dataset)

# 6. Training Loop

In [ ]:
# --- Quick Profile Check ---
print("Starting Profiling Check...")
train_with_profiling(model, train_loader, criterion, optimizer, scaler)

In [ ]:
wandb.init(
    project="GoogLeNet",
    config={
        "Architecture": "GoogLeNet",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMG_SIZE
    }
)

train_accuracies, test_accuracies = [], []
train_losses, val_losses = [], []

early_stopping = EarlyStopping(patience=10)

for epoch in range(EPOCHS):
    
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc = validate(model,  val_loader, criterion)
    scheduler.step(val_loss)

    test_acc = test(model, test_loader)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "lr": optimizer.param_groups[0]['lr']
    })

    print(f"Epoch: {epoch+1}/{EPOCHS} - train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}, " + 
           f"train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}, test_acc: {test_acc:.4f}")

    early_stopping(val_loss, model)

    if early_stopping.earlystop:
        print("Early Stopping")
        break

wandb.finish()

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label = "train_loss")
plt.plot(val_losses, label = "val_loss")
plt.title("Loss History")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label = "train_acc")
plt.plot(test_accuracies, label = "test_acc")
plt.title("Accuracy History")
plt.legend()

# 7. GradCAM

In [ ]:
!pip install grad-cam -q
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import random

# Pick the last Inception Block to setup the GradCAM
# In GoogLeNet, feature maps are often taken from one of the last inception modules
target_layers = [model.b5]

cam = GradCAM(model=model, target_layers=target_layers)

# Pick a random image form the test set
imgs, labels = next(iter(test_loader))
index = random.randint(0, len(imgs) - 1)
input_tensor = imgs[index].unsqueeze(0).to(device)
label = labels[index].item()

# Generate the Grad-CAM heatmap
grayscale_cam = cam(input_tensor=input_tensor, targets=None)
grayscale_cam = grayscale_cam[0, :]

# Prepare the image for Visualization
# create a 1D Tensor from the tuple of per-channel means/std
# and reshape it to (3, 1, 1) because imgs[index] has (3, H, W)
mean = torch.tensor(stats[0]).view(3, 1, 1)
std = torch.tensor(stats[1]).view(3, 1, 1)
rgb_img = imgs[index] * std + mean # Denormalize
rgb_img = rgb_img.permute(1, 2, 0).numpy()
rgb_img = np.clip(rgb_img, 0, 1)

# Overlay the heatmap on the image
visual = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# Get the Prediction
model.eval()
with torch.no_grad():
    out = model(input_tensor)
    pred = out.argmax(1).item()

# 7. Plotting
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title(f"Original (Class: {label})")
plt.imshow(rgb_img)
plt.axis('off')
plt.subplot(1, 2, 2)
plt.title(f"Grad-CAM (Pred: {pred})")
plt.imshow(visual)
plt.axis('off')
plt.tight_layout()
plt.show()